# 02 — Diagnóstico de Esquemas y Preparación de Reconstrucción

Este notebook corresponde a la **sección de Persona 1** del diagnóstico. Compara esquemas reales contra esquemas esperados y deja preparados los JSON de metadatos para Persona 2.

In [1]:
%pip install -q pyspark pyyaml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sys
import json
from pathlib import Path

# Ruta del proyecto
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Agregar carpeta src al path
sys.path.append(str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

# Configuración necesaria para PySpark en Windows
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("Proyecto:", PROJECT_ROOT)
print("Python usado por Spark:", sys.executable)
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

Proyecto: c:\Users\saray\Downloads\etl_spark_parquet_advanced_persona1
Python usado por Spark: c:\Users\saray\AppData\Local\Programs\Python\Python314\python.exe
HADOOP_HOME: C:\hadoop


In [3]:
from utils import load_config, get_spark_session
from schema_recovery import diagnose_successful_files, write_schema_differences

config = load_config("config/etl_config.yaml")
spark = get_spark_session(config)
spark.sparkContext.setLogLevel("WARN")

## 1. Cargar inventario técnico generado por Persona 1

In [4]:
audit_path = Path(config["paths"]["audit"]) / "audit_file_inventory"
inventory_df = spark.read.parquet(str(audit_path))
inventory_df.show(30, truncate=80)

+------------------------------------+----------------------+------------+-----------------------------------+--------------------------------------------------------------------------------+------------+--------------+---------------+------------+------------+------------+----------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------+
|                          process_id|         source_system|service_type|                          file_name|                                                                       file_path|file_size_mb|partition_year|partition_month| read_status|record_count|column_count|                                                     schema_hash|                                                                   error_message|              processed_at|
+------------------------------------+----------------------+------------+------------------------------

## 2. Mostrar esquemas esperados creados en metadata/

In [5]:
metadata_files = [
    "metadata/expected_schema_yellow.json",
    "metadata/expected_schema_green.json",
    "metadata/expected_schema_fhvhv.json",
    "metadata/canonical_schema_trips.json",
    "metadata/business_rules.json",
]

for file in metadata_files:
    print("\n===", file, "===")
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(json.dumps(data, indent=2, ensure_ascii=False)[:1500])


=== metadata/expected_schema_yellow.json ===
{
  "service_type": "yellow",
  "description": "Esquema esperado aproximado para archivos NYC TLC Yellow Taxi Trip Data.",
  "fields": [
    {
      "name": "VendorID",
      "type": "long",
      "nullable": true
    },
    {
      "name": "tpep_pickup_datetime",
      "type": "timestamp",
      "nullable": true
    },
    {
      "name": "tpep_dropoff_datetime",
      "type": "timestamp",
      "nullable": true
    },
    {
      "name": "passenger_count",
      "type": "double",
      "nullable": true
    },
    {
      "name": "trip_distance",
      "type": "double",
      "nullable": true
    },
    {
      "name": "RatecodeID",
      "type": "double",
      "nullable": true
    },
    {
      "name": "store_and_fwd_flag",
      "type": "string",
      "nullable": true
    },
    {
      "name": "PULocationID",
      "type": "integer",
      "nullable": true
    },
    {
      "name": "DOLocationID",
      "type": "integer",
      "nul

## 3. Comparación de esquema real vs esquema esperado

Se generan tres tipos de diferencia: `EXTRA_COLUMN`, `MISSING_COLUMN` y `TYPE_MISMATCH`.

In [6]:
process_id = inventory_df.select("process_id").first()[0]
schema_diff_df = diagnose_successful_files(spark, config, inventory_df, process_id)

schema_diff_df.show(100, truncate=80)
print("Total de diferencias detectadas:", schema_diff_df.count())

+------------------------------------+------------+-------------------------------+----------------------------------------------------------------+---------------+--------------+-------------+-----------+--------------------------+
|                          process_id|service_type|                      file_name|                                                     schema_hash|    column_name|     diff_type|expected_type|actual_type|             diagnostic_at|
+------------------------------------+------------+-------------------------------+----------------------------------------------------------------+---------------+--------------+-------------+-----------+--------------------------+
|7b8f423c-0032-4840-a69c-851fc0f6df92|      yellow|yellow_tripdata_2020-01.parquet|b22f96c0df88a74e907d88e844b9aca242de41060436af2589a785d0adb224bf|   PULocationID| TYPE_MISMATCH|      integer|     bigint|2026-06-14 07:04:21.883661|
|7b8f423c-0032-4840-a69c-851fc0f6df92|      yellow|yellow_tripdata_2

In [7]:
schema_diff_path = write_schema_differences(schema_diff_df, config)
print("schema_differences escrito en:", schema_diff_path)

schema_differences escrito en: C:\Users\saray\Downloads\etl_spark_parquet_advanced_persona1\data\audit\schema_differences


## 4. Resumen de diferencias por servicio

In [8]:
schema_diff_df.groupBy("service_type", "diff_type").count().orderBy("service_type", "diff_type").show(truncate=False)

+------------+--------------+-----+
|service_type|diff_type     |count|
+------------+--------------+-----+
|fhvhv       |TYPE_MISMATCH |2    |
|green       |TYPE_MISMATCH |8    |
|yellow      |EXTRA_COLUMN  |3    |
|yellow      |MISSING_COLUMN|3    |
|yellow      |TYPE_MISMATCH |20   |
+------------+--------------+-----+



## 5. Hashes de esquema por servicio

Permite identificar cambios estructurales entre meses o versiones del mismo tipo de servicio.

In [9]:
inventory_df.filter("read_status = 'SUCCESS'").groupBy(
    "service_type", "schema_hash"
).count().orderBy("service_type", "count").show(100, truncate=False)

+------------+----------------------------------------------------------------+-----+
|service_type|schema_hash                                                     |count|
+------------+----------------------------------------------------------------+-----+
|bad_parquet |9496c71d2688636e539adabcf05402ebe2cb4e8002af7e31f99414c54dfe2584|1    |
|bad_parquet |17f8760974e489fc41aef780b99fc8f0f58170236bb0db4d6f809035c088757e|1    |
|bad_parquet |124c6e2212380ecb4d9af060862fc4b08f8c6c36b5d2560bc0eaa770e6fbe5db|1    |
|bad_parquet |ca38d91ae51900dbc2b321120ce6f3c7d3bfc6ebf4e7584463eb431af5e13ba9|1    |
|fhvhv       |91998fe871a69b2ac2248af414c37e6128aea249ce6697b4ed1c3f1567b8d8ce|1    |
|green       |9a298f0f580f5862c876e0cada3dd5ff5d886acad3ce0026d39ffa47a7601525|1    |
|green       |ffd973ec83f6932081c7e2e4645283696cea4ca8cc7e5a4d92617b212d0c11d9|1    |
|yellow      |b22f96c0df88a74e907d88e844b9aca242de41060436af2589a785d0adb224bf|1    |
|yellow      |07f8a4ce182974ff28a1ff561f073c24258993bc